---
title: Interactive widgets and animations
short_title: "Day 4: interactives"
---

# Interactive widgets and animations

Interactive widgets rely on the `ipywidgets` package. In addition, we need to switch the `matplotlib` backend from `inline` to `ipympl`:

In [ ]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
import ipympl

In [ ]:
%matplotlib ipympl

As for animation, the functionality is built-in to `matplotlib`, in the `animation` module. In addition, let's import jupyter notebook's `HTML` display capabilities:

In [ ]:
from matplotlib import animation
import IPython.display as IDisplay

Also import some familiar packages

In [ ]:
import numpy as np
import scipy.integrate as sint

## Interactive widgets

Let's say we want students to explore how initial value affect the prey-predator dynamics. The basic idea is to separate the code into a "setup" part (runs once) and a "callback" part (runs everytime the user interact with the widgets)

In [ ]:
# setup code

def predator_prey(t, Y):
    return [
        alpha * Y[0] - beta * Y[0] * Y[1],
        -gamma * Y[1] + delta * Y[0] * Y[1]
    ]

alpha = 24
beta = 0.4
gamma = 2.0
delta = 0.002

t_span = [0, 5]
t_array = np.linspace(0, 5, 400)

In [ ]:
# Simple widgets via widgets.interact()

fig = plt.figure(figsize=(10, 4))
ax1 = fig.add_subplot(2, 2, 1)
ax2 = fig.add_subplot(2, 2, 3)
ax3 = fig.add_subplot(2, 2, (2, 4))

plt.show(fig)

def eval_prey_predator(prey_init, pred_init):

    out = sint.solve_ivp(
        predator_prey, t_span, [prey_init, pred_init], 
        t_eval=t_array, method="Radau"
    )
    
    Y_array = out.y

    ax1.cla()
    ax1.plot(t_array, Y_array[0], c="tab:blue", ls="-", label="prey")
    ax1.set_ylim(0, 15000)
    
    ax2.cla()
    ax2.plot(t_array, Y_array[1], c="tab:orange", ls="--", label="predator")
    ax2.set_ylim(0, 200)

    ax3.cla()
    ax3.scatter([prey_init], [pred_init], c="black", s=10)
    ax3.plot(Y_array[0], Y_array[1], c="tab:purple")
    ax3.set_xlim(0, 15000)
    ax3.set_ylim(0, 200)

widgets.interact(
    eval_prey_predator,
    prey_init = widgets.IntSlider(
        min=100, max=10000, step=100, value=3000, 
        description="initial prey density", style={'description_width': 'initial'}
    ),
    pred_init = widgets.IntSlider(
        min=10, max=100, step=1, value=30, 
        description="initial predator density", style={'description_width': 'initial'}
    )
);

In [ ]:
# even more control: VBox(), HBox(), Output(), and interactive_output()
# Also: replacing data as opposed to re-initiating plots

fig = plt.figure(figsize=(10, 4))
ax1 = fig.add_subplot(2, 2, 1)
ax2 = fig.add_subplot(2, 2, 3)
ax3 = fig.add_subplot(2, 2, (2, 4))

lines1 = ax1.plot(t_array, np.zeros_like(t_array), c="tab:blue", ls="-", label="prey")
ax1.set_ylim(0, 15000)

lines2 = ax2.plot(t_array, np.zeros_like(t_array), c="tab:orange", ls="--", label="predator")
ax2.set_ylim(0, 200)

sc = ax3.scatter([0], [0], c="black", s=10)
lines3 = ax3.plot([0], [0], c="tab:purple")
ax3.set_xlim(0, 15000)
ax3.set_ylim(0, 200)

prey_widget = widgets.IntSlider(description="prey", min=100, max=10000, step=100, value=3000)
pred_widget = widgets.IntSlider(description="predator", min=10, max=100, step=1, value=30)
output = widgets.Output(layout={'border': '1px solid black'})

with output:
    plt.show(fig)

inputs = widgets.HBox([prey_widget, pred_widget])
ui = widgets.VBox([output, inputs])

def eval_prey_predator(prey_init, pred_init):

    out = sint.solve_ivp(
        predator_prey, t_span, [prey_init, pred_init], 
        t_eval=t_array, method="Radau"
    )
    
    Y_array = out.y

    lines1[0].set_data(t_array, Y_array[0])
    lines2[0].set_data(t_array, Y_array[1])
    lines3[0].set_data(Y_array[0], Y_array[1])
    sc.set_offsets([[prey_init, pred_init]])

widgets.interactive_output(
    eval_prey_predator,
    {"prey_init": prey_widget, "pred_init": pred_widget}
)

display(ui)

For a list of widgets, consult the [widget list](https://ipywidgets.readthedocs.io/en/stable/examples/Widget%20List.html) from `ipywidgets`.

## Animation

The basic code structure of an animation is similar to the code structure of an interactive: codes need to be separated into an setup part (runs once) and a per-frame part (runs every frame)

In [ ]:
# Function to build a Leslie matrix given per-age-class birthrate and per-age-class mortality

def make_Leslie_Matrix(F, P, G):
    M = np.diag(P) + np.diag(G, -1)
    M[0, 1:] = F
    return M

In [ ]:
F = [0, 0, 0, 127, 4, 80]
P = [0, 0.7370, 0.6610, 0.6907, 0, 0, 0.8089]
G = [0.6757, 0.0486, 0.0147, 0.0518, 0.8091, 0.8091]

L = make_Leslie_Matrix(F, P, G) / 0.95

In [ ]:
N_now = np.array([0, 0, 0, 0, 5, 0, 0])

fig = plt.figure()
ax = fig.add_subplot()
ax.set_ylim(0, 1000)
#ax.set_ylim(1, 1000)
#ax.set_yscale("log", base=10)
ax.set_xlabel("Stage")
ax.set_ylabel("Individuals per area")
bars = ax.bar(np.arange(1, 8), N_now)

def init():
    ax.set_title(f"Population structure at year 1")
    for bar, val in zip(bars, N_now):
        bar.set_height(val)
    return bars.patches

def animate(i):
    global N_now
    if i > 0:
        N_now = L @ N_now
    ax.set_title(f"Population structure at year {max(1, i + 1)}")
    for bar, val in zip(bars, N_now):
        bar.set_height(val)
    return bars.patches
    

anim = animation.FuncAnimation(
    fig, animate, init_func = init, frames=40, interval=500, repeat=False
)

# saving animation to mp4
#anim.save('Leslie_animation.mp4', writer = 'ffmpeg', fps=2)

# save a base64 string of html
#html_vid = anim.to_html5_video()
#with open("Leslie_animation_vid.html", "w") as outfile:
#    outfile.write(html_vid)

# save a base64 string of html (alternative format)
#html_js = anim.to_jshtml()
#with open("Leslie_animation_js.html", "w") as outfile:
#    outfile.write(html_js)

## Exercise: van der Pol redux

Build an interactive or animination illustrating how the phase portrait of the van der Pol oscillator changes with $\mu$